In [1]:
# con.execute("some SQL here")          # run a command, no result needed
# con.execute("SELECT...").fetchall()   # run and get all results as a list
# con.execute("SELECT...").df()         # run and get results as a Pandas dataframe
# con.execute("SELECT...").fetchone()   # run and get just the first result

In [2]:
from datetime import datetime, timedelta
import requests
import pytz
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import duckdb
import pandas as pd
import time

analyzer = SentimentIntensityAnalyzer()
eastern = pytz.timezone('America/Toronto')

before_date = datetime.now().strftime("%Y-%m-%d")
after_date = (datetime.now() - timedelta(days=180)).strftime("%Y-%m-%d")

print(f"Pulling data from {after_date} to {before_date}")

Pulling data from 2025-09-27 to 2026-03-26


In [3]:
con = duckdb.connect('/Users/rimona/fintech-pulse/fintech_pulse.duckdb')

con.execute("""
    CREATE TABLE IF NOT EXISTS raw_posts (
        post_id TEXT PRIMARY KEY,
        company TEXT,
        subreddit TEXT,
        title TEXT,
        body TEXT,
        upvote_score INTEGER,
        upvote_ratio FLOAT,
        comment_count INTEGER,
        sentiment_score FLOAT,
        created_utc TIMESTAMP,
        loaded_at TIMESTAMP,
        updated_at TIMESTAMP,
        content_type TEXT
    )
""")

print("Database and table ready")
con.execute("DESCRIBE raw_posts").df()

Database and table ready


,column_name,column_type,null,key,default,extra
0,post_id,VARCHAR,NO,PRI,None,None
1,company,VARCHAR,YES,None,None,None
2,subreddit,VARCHAR,YES,None,None,None
3,title,VARCHAR,YES,None,None,None
4,body,VARCHAR,YES,None,None,None
5,upvote_score,INTEGER,YES,None,None,None
6,upvote_ratio,FLOAT,YES,None,None,None
7,comment_count,INTEGER,YES,None,None,None
8,sentiment_score,FLOAT,YES,None,None,None
9,created_utc,TIMESTAMP,YES,None,None,None


In [4]:
def fetch_by_field(company, subreddit, after_date, before_date, field):
    
    url = "https://arctic-shift.photon-reddit.com/api/posts/search"
    headers = {"User-Agent": "fintech-pulse:v1.0 (by Rimona)"}
    
    all_posts = {}
    current_after = after_date
    
    while True:
        params = {
            "subreddit": subreddit,
            field: company,
            "after": current_after,
            "before": before_date,
            "limit": 100,
            "sort": "asc"
        }
        
        response = requests.get(url, params=params, headers=headers)
        time.sleep(2)
        data = response.json()
        
        if not data.get('data'):
            break
        
        for item in data['data']:
            text = item.get('title', '') + " " + item.get('selftext', '')
            sentiment_score = analyzer.polarity_scores(text)['compound']
            created_utc = datetime.fromtimestamp(item['created_utc'], tz=eastern)
            
            post_id = item['id']
            all_posts[post_id] = {
                'post_id': post_id,
                'company': company,
                'subreddit': subreddit,
                'title': item.get('title', ''),
                'body': item.get('selftext', ''),
                'upvote_score': item.get('score', 0),
                'upvote_ratio': item.get('upvote_ratio', 0),
                'comment_count': item.get('num_comments', 0),
                'sentiment_score': sentiment_score,
                'created_utc': created_utc,
                'content_type': 'post'
            }
        
        if len(data['data']) < 100:
            break
        
        last_utc = data['data'][-1]['created_utc']
        current_after = str(last_utc + 1)
        
        print(f"Got {len(data['data'])} posts by {field}, paginating from {last_utc}...")
    
    return list(all_posts.values())

In [5]:
def fetch_posts(company, subreddit, after_date, before_date):
    
    print(f"Searching posts by title for {company}...")
    by_title = fetch_by_field(company, subreddit, after_date, before_date, field="title")
    
    print(f"Searching posts by body for {company}...")
    by_body = fetch_by_field(company, subreddit, after_date, before_date, field="selftext")
    
    # Combine and deduplicate by post_id
    combined = {}
    for post in by_title + by_body:
        combined[post['post_id']] = post
    
    result = list(combined.values())
    print(f"Total unique posts found: {len(result)}")
    return result

In [6]:
def fetch_comments(company, subreddit, after_date, before_date):
    
    url = "https://arctic-shift.photon-reddit.com/api/comments/search"
    headers = {"User-Agent": "fintech-pulse:v1.0 (by Rimona)"}
    
    all_comments = {}
    current_after = after_date
    
    while True:
        params = {
            "subreddit": subreddit,
            "body": company,
            "after": current_after,
            "before": before_date,
            "limit": 100,
            "sort": "asc"
        }
        
        response = requests.get(url, params=params, headers=headers)
        time.sleep(2)
        data = response.json()
        
        if not data.get('data'):
            break
        
        for item in data['data']:
            text = item.get('body', '')
            sentiment_score = analyzer.polarity_scores(text)['compound']
            created_utc = datetime.fromtimestamp(item['created_utc'], tz=eastern)
            
            comment_id = item['id']
            all_comments[comment_id] = {
                'post_id': comment_id,
                'company': company,
                'subreddit': subreddit,
                'title': '',
                'body': item.get('body', ''),
                'upvote_score': item.get('score', 0),
                'upvote_ratio': item.get('upvote_ratio', 0),
                'comment_count': 0,
                'sentiment_score': sentiment_score,
                'created_utc': created_utc,
                'content_type': 'comment'
            }
        
        if len(data['data']) < 100:
            break
        
        last_utc = data['data'][-1]['created_utc']
        current_after = str(last_utc + 1)
        
        print(f"Got {len(data['data'])} comments, paginating from {last_utc}...")
    
    return list(all_comments.values())

In [7]:
def insert_posts(results):
    
    if len(results) == 0:
        print("No results found, skipping")
        return
    
    now = datetime.now()
    
    for item in results:
        con.execute("""
            INSERT OR REPLACE INTO raw_posts (
                post_id,
                company,
                subreddit,
                title,
                body,
                upvote_score,
                upvote_ratio,
                comment_count,
                sentiment_score,
                created_utc,
                loaded_at,
                updated_at,
                content_type
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, [
            item['post_id'],
            item['company'],
            item['subreddit'],
            item['title'],
            item['body'],
            item['upvote_score'],
            item['upvote_ratio'],
            item['comment_count'],
            item['sentiment_score'],
            item['created_utc'],
            now,
            now,
            item['content_type']
        ])
    
    print(f"Inserted {len(results)} {results[0]['content_type']}s for {results[0]['company']} from r/{results[0]['subreddit']}")

In [8]:
companies = [
    "Wealthsimple", "Koho", "Questrade",
  "EQ Bank", "Neo Financial", "Mogo", "Shakepay", "Float Financial"
]

subreddits = [
    "PersonalFinanceCanada",
    "CanadaFinance",
    "investing_canada"
]

for company in companies:
    for subreddit in subreddits:
        print(f"\n--- Fetching {company} from r/{subreddit} ---")
        
        posts = fetch_posts(company, subreddit, after_date, before_date)
        insert_posts(posts)
        
        comments = fetch_comments(company, subreddit, after_date, before_date)
        insert_posts(comments)

print("\nHistorical load complete!")
con.execute("SELECT content_type, COUNT(*) as count FROM raw_posts GROUP BY content_type").df()


--- Fetching Wealthsimple from r/PersonalFinanceCanada ---
Searching posts by title for Wealthsimple...
Got 100 posts by title, paginating from 1763305526...
Got 100 posts by title, paginating from 1767311625...
Got 100 posts by title, paginating from 1771277677...
Searching posts by body for Wealthsimple...
Got 100 posts by selftext, paginating from 1760102921...
Got 100 posts by selftext, paginating from 1761762835...
Got 100 posts by selftext, paginating from 1763950188...
Got 100 posts by selftext, paginating from 1765554875...
Got 100 posts by selftext, paginating from 1767038934...
Got 100 posts by selftext, paginating from 1767918351...
Got 100 posts by selftext, paginating from 1769182786...
Got 100 posts by selftext, paginating from 1770726423...
Got 100 posts by selftext, paginating from 1772312726...
Got 100 posts by selftext, paginating from 1773605155...
Total unique posts found: 1202
Inserted 1202 posts for Wealthsimple from r/PersonalFinanceCanada
Got 100 comments, pagi

,content_type,count
0,comment,7061
1,post,1590
